# An AI agent that chases overdue invoices

**Bloomfield Bakery Ltd** supplies 15 cafés, hotels and shops around the Midlands. Like most small businesses, it has overdue invoices to chase, and chasing them well needs more than a list of what's past its due date. Who already paid without quoting the invoice number? Who's on a payment plan? Who's disputing a delivery?

This notebook shows the same job done three ways:

1. **A workflow**: a fixed rule, the same steps every time.
2. **An agent**: an AI model that reads the files, writes its own code, checks the results, and decides what to do.
3. **The leash**: the agent drafts the reminders, but nothing is sent until the owner approves.

Each approach is scored against **11 test cases** with known correct answers.

*Bloomfield Bakery, its customers and all the figures are fictional.* Part of a LinkedIn series by Declan Lenahan, DL Consultancy.

**Before you start:** Runtime › Change runtime type › **T4 GPU**. It works without a GPU, but uses a smaller model and runs slowly. Then run the cells from top to bottom. You don't need to write any code.

In [ ]:
#@title Setup: run this first (about 5 minutes the first time)
#@markdown Installs the agent framework, loads the bakery's files and the AI model.
import importlib.metadata as _md, subprocess, sys

def _version(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

_install = []
if _version("smolagents") != "1.26.0":
    _install.append("smolagents==1.26.0")
if _version("transformers") is None or int(_version("transformers").split(".")[0]) < 5:
    _install.append("transformers==5.18.0")
import torch
if torch.cuda.is_available() and _version("bitsandbytes") is None:
    _install.append("bitsandbytes==0.50.2")
if _install:
    print("Installing", ", ".join(_install), "...", flush=True)
    _r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_install], capture_output=True, text=True)
    if _r.returncode != 0:
        print(_r.stderr[-2000:])
        raise RuntimeError("Installation failed. Runtime › Restart session, then run Setup again.")

import os, re, ast, time, html, io, warnings, logging, urllib.request
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import pandas as pd, transformers
from IPython.display import display, HTML
transformers.logging.set_verbosity_error()
import smolagents
from smolagents import CodeAgent, TransformersModel, tool
from smolagents.memory import ActionStep, FinalAnswerStep
from smolagents.utils import AgentGenerationError, AgentMaxStepsError
logging.getLogger("smolagents").setLevel(logging.ERROR)

# ------------------------------------------------------------ the bakery's files
AS_OF = pd.Timestamp("2026-11-02")       # "today" in the scenario
DATA_URL = "https://raw.githubusercontent.com/Declan444/invoice-chasing-agent/main/data/"   # change if the repo moves
FILES = ["invoices.csv", "bank_statement.csv", "customer_notes.csv", "answer_key.csv"]

def _load(name):
    if os.path.exists(name):                                  # a copy in the Colab folder wins
        return pd.read_csv(name)
    try:
        with urllib.request.urlopen(DATA_URL + name, timeout=60) as r:
            return pd.read_csv(io.BytesIO(r.read()))
    except Exception:
        from google.colab import files as _files
        print(f"Could not download {name}. Please upload it (all four files at once is fine):")
        _files.upload()
        return pd.read_csv(name)

invoices, bank, notes, answer_key = (_load(f) for f in FILES)
invoices["date"] = pd.to_datetime(invoices["date"])
invoices["due_date"] = pd.to_datetime(invoices["due_date"])
invoices["applies_to"] = invoices["applies_to"].fillna("")
bank["date"] = pd.to_datetime(bank["date"])
notes["note_date"] = pd.to_datetime(notes["note_date"])
print(f"Bloomfield Bakery: {(invoices.type == 'Invoice').sum()} invoices, {len(bank)} bank lines, "
      f"{len(notes)} customer notes, {len(answer_key)} test cases.")

# ------------------------------------------------------------ the model
GPU = torch.cuda.is_available()
if GPU and torch.cuda.get_device_capability(0)[0] < 8:
    # T4 workaround from the course notebook: keeps memory-efficient attention on long agent prompts
    import transformers.integrations.sdpa_attention as _sdpa
    if hasattr(_sdpa, "use_gqa_in_sdpa"):
        _sdpa.use_gqa_in_sdpa = lambda attention_mask, key, value: False
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct" if GPU else "Qwen/Qwen2.5-Coder-1.5B-Instruct"
OUTPUT_LIMIT = 3000      # how much of each printed result the model reads back
import smolagents.agents as _sa, smolagents.utils as _su
_sa.truncate_content = lambda content, max_length=OUTPUT_LIMIT: _su.truncate_content(content, max_length)

FENCE = re.compile(r"```(?:python|py)[ \t]*\n(.*?)(?:\n[ \t]*```|$)", re.S)
FINAL_PROSE = re.compile(r"(?:\*\*)?final answer(?:\*\*)?\s*:\s*(.+)", re.I | re.S)

def _is_python(code):
    try:
        ast.parse(code); return True
    except SyntaxError:
        return False

def repair_output(text):
    """If the model writes 'Final Answer: ...' as words instead of code, turn it into the code smolagents expects."""
    text = text or ""
    block = FENCE.search(text)
    if block and _is_python(block.group(1)):
        return text
    words = FINAL_PROSE.search(block.group(1) if block else text)
    if not words:
        return text
    answer = words.group(1).strip().strip("`*").strip()
    before = text[:block.start()] if block else text[:words.start()]
    return before.rstrip() + "\n```python\nfinal_answer(" + repr(answer) + ")\n```"

class AgentModel(TransformersModel):
    """The model, stopped as soon as it closes its block of Python (otherwise it keeps explaining and runs out of tokens)."""
    def make_stopping_criteria(self, stop_sequences, tokenizer):
        from transformers import StoppingCriteria, StoppingCriteriaList
        closed = re.compile(r"```(?:python|py)[ \t]*\n.*?\n[ \t]*```", re.S)
        stops = list(stop_sequences or [])
        class StopAfterCode(StoppingCriteria):
            def __init__(self): self.text = ""
            def __call__(self, input_ids, scores, **kwargs):
                self.text += tokenizer.decode(input_ids[0][-1], skip_special_tokens=True)
                done = bool(closed.search(self.text)) or any(self.text.endswith(s) for s in stops)
                return torch.full((input_ids.shape[0],), done, dtype=torch.bool, device=input_ids.device)
        return StoppingCriteriaList([StopAfterCode()])

    def generate(self, messages, stop_sequences=None, response_format=None, tools_to_call_from=None, **kwargs):
        m = super().generate(messages, stop_sequences=stop_sequences, response_format=response_format,
                             tools_to_call_from=tools_to_call_from, **kwargs)
        m.content = repair_output(m.content)
        return m

if "_llm" not in globals():
    kw = {}
    if GPU:
        from transformers import BitsAndBytesConfig
        kw["model_kwargs"] = {"quantization_config": BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                                        bnb_4bit_compute_dtype=torch.float16)}
    print(f"Loading {MODEL_ID} (the first time includes the download) ...", flush=True)
    t = time.time()
    _llm = AgentModel(model_id=MODEL_ID, device_map="cuda" if GPU else "cpu",
                      torch_dtype="float16" if GPU else "bfloat16", max_new_tokens=900, do_sample=False, **kw)
    print(f"Model ready in {time.time() - t:.0f} s.")

# ------------------------------------------------------------ display helpers
def box(label, text, colour="#EEF1F5", mono=False):
    font = "font-family:monospace;font-size:13px;" if mono else "font-family:sans-serif;font-size:14px;"
    display(HTML(f"<div style='{font}background:{colour};color:#0F1B2D;border-radius:8px;padding:8px 12px;margin:6px 0;white-space:pre-wrap'>"
                 f"<b style='font-family:sans-serif;color:#0E6B6A;font-size:12px'>{html.escape(label.upper())}</b><br>{text}</div>"))

def show(df):
    display(df.style.hide(axis="index").format(precision=2, escape="html")
            .set_properties(**{"text-align": "left", "vertical-align": "top"})
            .set_table_styles([{"selector": "th", "props": [("background-color", "#0F1B2D"), ("color", "white"),
                                                             ("text-align", "left"), ("padding", "4px 8px")]},
                               {"selector": "td", "props": [("padding", "4px 8px")]}]))

# ------------------------------------------------------------ scoring against the answer key
def _key_cases():
    cases = []
    for _, r in answer_key.iterrows():
        act = r["correct_action"]
        tone = next((t for t in ("gentle", "standard", "firm") if t in act.lower()), None)
        cases.append(dict(case=int(r["case"]), invoices=[s.strip() for s in r["document_no"].split(",")],
                          customer=r["customer_name"], chase=act.startswith("CHASE"), tone=tone,
                          amount=float(str(r["amount_to_chase"]).replace("€", "").replace(",", "")), action=act))
    return cases

def score(reminders, title):
    """reminders: list of dicts with invoice_numbers (list), amount_eur, tone. Everything else counts as 'not chased'."""
    rows, right = [], 0
    for c in _key_cases():
        hits = [r for r in reminders if set(r["invoice_numbers"]) & set(c["invoices"])]
        if c["chase"]:
            ok_amount = bool(hits) and abs(hits[0]["amount_eur"] - c["amount"]) <= 1.0
            ok = ok_amount
            did = f"Chased €{hits[0]['amount_eur']:,.2f} ({hits[0]['tone']})" if hits else "Not chased"
            tone_ok = "" if not (hits and c["tone"]) else ("tone right" if hits[0]["tone"] == c["tone"] else f"tone: wanted {c['tone']}")
        else:
            ok = not hits
            did = f"Chased €{hits[0]['amount_eur']:,.2f}" if hits else "Not chased"
            tone_ok = ""
        right += ok
        rows.append({"Case": c["case"], "Customer": c["customer"], "Should": c["action"], "Did": did,
                     "Result": "✅" if ok else "❌", "Note": tone_ok})
    keyed = {i for c in _key_cases() for i in c["invoices"]}
    extra = [r for r in reminders if not set(r["invoice_numbers"]) & keyed]
    box(title, f"<b style='font-size:20px'>{right} of 11 right</b>"
        + (f"<br>Plus {len(extra)} reminder(s) for invoices that are not overdue at all." if extra else ""), "#E2F1F0")
    show(pd.DataFrame(rows))
    return right

print(f"Setup is done. Running on {'GPU: ' + torch.cuda.get_device_name(0) if GPU else 'CPU (slow)'}.")

## The bakery's files

These are the three files the owner has, and the same three the agent gets:

- **Invoices**: exported from the accounts system, with the due date the system calculated.
- **Bank statement**: money in and out. Some customers quote the invoice number, some don't.
- **Customer notes**: what normally lives in the owner's head, written down.

In [ ]:
#@title Look at the files
#@markdown Pick a file to see it in full.
file = "Customer notes"  #@param ["Invoices", "Bank statement", "Customer notes"]
df = {"Invoices": invoices, "Bank statement": bank, "Customer notes": notes}[file].copy()
for c in ["date", "due_date", "note_date"]:
    if c in df:
        df[c] = df[c].dt.strftime("%d %b %Y").fillna("")
show(df)

## Act 1: the workflow

The simplest approach: **chase every invoice past its due date that has no payment quoting its invoice number.** Same rule, same steps, every time. No AI involved.

This is how a lot of businesses (and a lot of software) do it. Let's see how it scores.

In [ ]:
#@title Run the workflow and score it
paid = set(re.findall(r"INV-\d+", " ".join(bank["description"].astype(str))))
overdue = invoices[(invoices.type == "Invoice") & (invoices.due_date < AS_OF) & (~invoices.document_no.isin(paid))]
workflow_reminders = [dict(invoice_numbers=[r.document_no], amount_eur=r.amount_eur, tone="standard",
                           customer=r.customer_name) for r in overdue.itertuples()]
box("The workflow's code (the whole thing)",
    html.escape('paid = invoice numbers quoted on the bank statement\n'
                'chase every invoice where due_date < today and the number is not in paid'), "#FFFFFF", mono=True)
print(f"The workflow would send {len(workflow_reminders)} reminders.")
workflow_score = score(workflow_reminders, "The workflow")

**3 of 11.** The rule is fast, cheap and predictable, but it can't read. It doesn't know that O'Brien's paid without quoting the number, that the golf club has 60-day terms the accounts system was never updated with, or that the Bridge House Hotel is on a payment plan. It also misses that Rosie's Deli only part-paid, because her payment quoted the invoice number.

Every one of those answers was sitting in the files. Reading them needs judgement.

## Act 2: the agent

Now the AI model gets the same three files and a goal, the way you'd brief a new member of staff. It isn't told about any of the traps.

It works in a loop: **think, write a few lines of Python, run them, read the result, decide what to do next**, until it's done. Each step appears below as it happens.

It has two tools:

- `queue_reminder`: drafts a reminder for the owner to approve. **Nothing is sent.**
- `no_reminder`: records an invoice it decided *not* to chase, and why.

With a T4 GPU this takes a few minutes. Without one, it can take much longer.

In [ ]:
#@title The agent's tools, and its brief
QUEUE, SKIPPED = [], []

def _invoice_list(text):
    return re.findall(r"INV-\d+", str(text))

@tool
def queue_reminder(customer_name: str, invoice_numbers: str, amount_eur: float, tone: str, email_text: str) -> str:
    """Drafts a payment reminder email to a customer and puts it in a queue for the owner to approve. Nothing is sent until the owner approves it.

    Args:
        customer_name: the customer's name, as in the invoices table.
        invoice_numbers: the invoice numbers the reminder is about, comma-separated, for example "INV-2024".
        amount_eur: the amount the customer still owes on those invoices, in euro.
        tone: "gentle", "standard" or "firm".
        email_text: the full text of the email, at most six sentences.
    """
    QUEUE.append(dict(customer=customer_name, invoice_numbers=_invoice_list(invoice_numbers),
                      amount_eur=float(amount_eur), tone=str(tone).lower().strip(), email_text=email_text))
    return f"Queued for the owner's approval: reminder to {customer_name} for €{float(amount_eur):,.2f}. Nothing has been sent."

@tool
def no_reminder(customer_name: str, invoice_numbers: str, reason: str) -> str:
    """Records that an invoice that looks overdue should NOT get a reminder, and why, so the owner can see the decision.

    Args:
        customer_name: the customer's name, as in the invoices table.
        invoice_numbers: the invoice numbers, comma-separated.
        reason: one sentence on why no reminder should be sent, and anything the owner should do instead.
    """
    SKIPPED.append(dict(customer=customer_name, invoice_numbers=_invoice_list(invoice_numbers), reason=reason))
    return f"Recorded: no reminder for {customer_name}."

BRIEF = """You help the owner of Bloomfield Bakery Ltd chase overdue invoices. Today is 2 November 2026.

Three pandas DataFrames are already loaded. Use them directly, never read files:
- `invoices`: document_no, type ("Invoice" or "Credit note"), customer_id, customer_name, date, due_date, amount_eur, applies_to (for a credit note: the invoice it reduces). due_date is what the accounts system calculated.
- `bank`: date, description, money_in_eur, money_out_eur. Customer payments are money in. Some quote the invoice number in the description, some do not.
- `notes`: customer_id, customer_name, note_date, note. The owner's notes on customers. Notes with customer_id "ALL" are house rules for every customer.

Your job: find every invoice that looks overdue today, and decide for each customer whether to send a reminder.
Before deciding, check the bank statement for payments (with or without an invoice number) and read the owner's notes and house rules.
For each customer to chase, call queue_reminder once with the amount still owed and a fitting tone.
For each overdue-looking invoice you decide not to chase, call no_reminder with the reason.
Never send anything yourself: the owner approves every reminder. When you have decided on every customer, call final_answer with a short summary.

End every code block with print() of what you need to see. An observation of None only means nothing was printed."""

AGENT_IMPORTS = ["pandas", "numpy", "re", "datetime"]
print("Tools and brief ready. Run the next cell to start the agent.")

In [ ]:
#@title Run the agent
max_steps = 12  #@param {type:"slider", min:4, max:20, step:1}
QUEUE.clear(); SKIPPED.clear()
agent = CodeAgent(tools=[queue_reminder, no_reminder], model=_llm, additional_authorized_imports=AGENT_IMPORTS,
                  max_steps=max_steps, verbosity_level=-1, code_block_tags="markdown",
                  max_print_outputs_length=OUTPUT_LIMIT)
start = {"invoices": invoices.copy(), "bank": bank.copy(), "notes": notes.copy(), "pd": pd}
t0, answer, seen = time.time(), None, set()
stream = agent.run(BRIEF, stream=True, additional_args=start)
try:
    for ev in stream:
        if isinstance(ev, ActionStep) and ev.step_number not in seen:
            seen.add(ev.step_number)
            n, sec = ev.step_number, (ev.timing.duration or 0.0)
            thought = re.split(r"```|<code>", ev.model_output if isinstance(ev.model_output, str) else "")[0]
            thought = thought.replace("Thought:", "").replace("Code:", "").strip()
            box(f"Step {n}: thinking ({sec:.0f} s)", html.escape(thought) or "<i>(no thought written)</i>", "#E2F1F0")
            if ev.code_action:
                box(f"Step {n}: the code it wrote", html.escape(ev.code_action), "#FFFFFF", mono=True)
            if ev.error:
                box(f"Step {n}: error (the agent reads this and tries again)", html.escape(str(ev.error)), "#FBE3E3", mono=True)
            elif ev.observations:
                box(f"Step {n}: what the computer printed back", html.escape(ev.observations.strip()), "#EEF1F5", mono=True)
        elif isinstance(ev, FinalAnswerStep):
            answer = ev.output
except AgentGenerationError as e:
    box("The model could not write its next step", html.escape(str(e)[:300]) + "<br>Run the cell again.", "#FBE3E3")
finally:
    stream.close()
    if GPU:
        torch.cuda.empty_cache()
if agent.memory.steps and isinstance(getattr(agent.memory.steps[-1], "error", None), AgentMaxStepsError):
    box("The agent ran out of steps", "It was cut off before finishing. Try more steps, or read where it got stuck.", "#FBE3E3")
box(f"The agent's summary, after {len(seen)} steps ({time.time() - t0:.0f} s)", f"<b>{html.escape(str(answer))}</b>", "#D5EBE9")
print(f"Reminders queued: {len(QUEUE)}. Invoices it decided not to chase: {len(SKIPPED)}.")

In [ ]:
#@title Score the agent against the 11 cases
agent_score = score(QUEUE, "The agent")
if SKIPPED:
    box("What it decided not to chase, and why", "", "#EEF1F5")
    show(pd.DataFrame([{"Customer": s["customer"], "Invoices": ", ".join(s["invoice_numbers"]), "Reason": s["reason"]}
                       for s in SKIPPED]))
box("Scoreboard", f"Workflow: <b>{workflow_score} of 11</b> &nbsp;·&nbsp; Agent ({MODEL_ID.split('/')[-1]}): <b>{agent_score} of 11</b>", "#E2F1F0")

**Read the code, not just the score.** For every decision, the steps above show which rows it looked at and what it calculated. That's the receipt. If it got a case wrong, scroll up and find the step where it went off track. Was it the data, the reasoning, or the brief?

Small open models are not consistent. Run it again and you may get a different score. That alone is worth knowing before you hand one your accounts.

## Act 3: the leash

The agent has drafted reminders, but **nothing has been sent**. This is the owner's job now: read each draft, then approve or reject it. In the research on levels of autonomy, this is level 4: *the user as approver*. The agent does the work, the person makes the call on anything that leaves the building.

In [ ]:
#@title Read the drafts waiting for approval
if not QUEUE:
    print("Nothing queued. Run the agent first.")
for i, r in enumerate(QUEUE, 1):
    box(f"Draft {i}: {r['customer']} · {', '.join(r['invoice_numbers'])} · €{r['amount_eur']:,.2f} · tone: {r['tone']}",
        html.escape(r["email_text"]), "#FFFFFF")

In [ ]:
#@title Approve or reject
#@markdown Type the draft numbers you approve, separated by commas (for example `1, 2, 4`). Leave empty to reject all.
approve = ""  #@param {type:"string"}
chosen = {int(x) for x in re.findall(r"\d+", approve)}
for i, r in enumerate(QUEUE, 1):
    status = "APPROVED: would be sent now" if i in chosen else "Rejected: not sent"
    print(f"Draft {i} to {r['customer']}: {status}")
print("\nIn this demo nothing is ever emailed. In a real business, only the approved drafts would go out.")

## The easy route: try it with Claude or ChatGPT

You don't need to build anything to try this on your own business. Download the three files (`invoices.csv`, `bank_statement.csv`, `customer_notes.csv`), upload them to Claude or ChatGPT, and paste this:

> I run Bloomfield Bakery. Today is 2 November 2026. These are my invoices, bank statement and notes on my customers. Find every invoice that looks overdue. Before deciding, check the bank statement for payments, with or without an invoice number, and read my notes and house rules. For each customer, tell me whether to send a reminder, for how much, and in what tone, and draft the email. For any invoice I shouldn't chase, tell me why. Show me the calculations you ran.

Then compare its answer with `answer_key.csv`. And open the code it ran: does it count what you meant?

---

**Credits.** Agent framework: [smolagents](https://github.com/huggingface/smolagents) 1.26.0 (Hugging Face, Apache 2.0). Model: Qwen2.5-Coder-7B-Instruct, or 1.5B without a GPU (Alibaba Qwen team, Apache 2.0). Model setup adapted from the Week 8 notebook of the Digital4Business Generative AI module (NOVA IMS). Data: fictional, generated by `make_data.py`.